# 🎬 Comprehensive Movie Dataset EDA & Clustering

In [1]:
import pandas as pd

# Load dataset
df = pd.read_csv("movie_cluster_data.csv")
df.head()


,is_Action,is_Comedy,is_Drama,is_Horror,is_Romance,is_Sci-Fi,popularity,runtime,vote_average,title,cluster
0,0,1,0,0,0,1,53.261330,126.965298,7.419076,The Broken Day of Justice,4
1,0,0,0,1,0,0,37.488864,109.612643,6.209725,The Broken Night of Time,2
2,0,0,1,0,1,1,59.240270,91.432315,6.767392,The Hidden City of Time,1
3,1,0,1,0,1,1,48.150979,108.588742,6.821698,The Lost Dream of Time,1
4,1,1,1,1,1,1,44.772770,120.990331,5.831910,The Glorious Revenge of Secrets,3


## 1. 📊 Single-Variable Analysis

In [2]:
import plotly.express as px

# Histograms
for col in ["popularity", "runtime", "vote_average"]:
    fig = px.histogram(df, x=col, nbins=20, title=f"Distribution of {col.title()}")
    fig.show()

# Boxplots by cluster
for col in ["popularity", "runtime", "vote_average"]:
    fig = px.box(df, x="cluster", y=col, points="all", title=f"{col.title()} by Cluster")
    fig.show()

# Genre proportions
genre_cols = [col for col in df.columns if col.startswith("is_")]
genre_mean = df[genre_cols].mean().sort_values()
fig = px.bar(x=genre_mean.index, y=genre_mean.values, title="Overall Genre Distribution")
fig.update_layout(xaxis_title="Genre", yaxis_title="Proportion")
fig.show()


## 2. 🔁 Bivariate Analysis

In [3]:
# Scatter matrix
fig = px.scatter_matrix(df, dimensions=["popularity", "runtime", "vote_average"],
                        color="cluster", title="Scatter Matrix of Key Features")
fig.show()

# Correlation heatmap
import numpy as np
import plotly.figure_factory as ff

corr = df[["popularity", "runtime", "vote_average"]].corr()
fig = ff.create_annotated_heatmap(
    z=np.round(corr.values, 2),
    x=list(corr.columns),
    y=list(corr.index),
    colorscale="Viridis",
    showscale=True
)
fig.update_layout(title="Correlation Heatmap")
fig.show()

# Scatter by pair with cluster color
fig = px.scatter(df, x="popularity", y="vote_average", color="cluster", hover_data=["title"],
                 title="Popularity vs. Vote Average by Cluster")
fig.show()


## 3. 🌐 Embedding and Clustering

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

# Feature selection and scaling
features = ["popularity", "runtime", "vote_average"] + [col for col in df.columns if col.startswith("is_")]
X = df[features]
X_scaled = StandardScaler().fit_transform(X)

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(X_scaled)
df["pca_x"], df["pca_y"] = pca_result[:, 0], pca_result[:, 1]

# t-SNE
tsne = TSNE(n_components=2, perplexity=30, learning_rate="auto", init="pca", random_state=42, n_iter=500)
tsne_result = tsne.fit_transform(X_scaled)
df["tsne_x"], df["tsne_y"] = tsne_result[:, 0], tsne_result[:, 1]

# Clustering
kmeans = KMeans(n_clusters=5, random_state=42)
df["cluster"] = kmeans.fit_predict(df[["tsne_x", "tsne_y"]])


c:\Users\xiucai admin\anaconda3\lib\site-packages\sklearn\manifold\_t_sne.py:982: FutureWarning:

The PCA initialization in TSNE will change to have the standard deviation of PC1 equal to 1e-4 in 1.2. This will ensure better convergence.



In [5]:
# Visualize PCA
fig = px.scatter(df, x="pca_x", y="pca_y", color="cluster", hover_data=["title"], title="PCA Projection")
fig.show()

# Visualize t-SNE
fig = px.scatter(df, x="tsne_x", y="tsne_y", color="cluster", hover_data=["title"], title="t-SNE Projection")
fig.show()


## 4. 🧠 Cluster Interpretation

In [6]:
for cluster_id in sorted(df["cluster"].unique()):
    subset = df[df["cluster"] == cluster_id]
    print(f"\n=== Cluster {cluster_id} ===")
    print(f"Size: {len(subset)}")
    print(f"Avg Rating: {subset['vote_average'].mean():.2f}")
    print(f"Avg Runtime: {subset['runtime'].mean():.1f} mins")
    print(f"Avg Popularity: {subset['popularity'].mean():.1f}")
    top_genres = subset[[c for c in df.columns if c.startswith("is_")]].mean().sort_values(ascending=False)
    print("Top Genres:")
    print(top_genres.head(3))
    print("Sample Titles:")
    print(subset['title'].head(3).to_string(index=False))



=== Cluster 0 ===
Size: 64
Avg Rating: 6.75
Avg Runtime: 103.0 mins
Avg Popularity: 51.5
Top Genres:
is_Horror     0.84375
is_Romance    0.81250
is_Drama      0.71875
dtype: float64
Sample Titles:
The Glorious Revenge of Secrets
       The Lost World of Escape
         The Lost Day of Future

=== Cluster 1 ===
Size: 56
Avg Rating: 6.42
Avg Runtime: 101.8 mins
Avg Popularity: 53.0
Top Genres:
is_Sci-Fi     1.000000
is_Romance    0.660714
is_Action     0.642857
dtype: float64
Sample Titles:
The Hidden City of Time
 The Lost Dream of Time
The Final Truth of Fire

=== Cluster 2 ===
Size: 64
Avg Rating: 6.42
Avg Runtime: 100.4 mins
Avg Popularity: 49.4
Top Genres:
is_Comedy    0.734375
is_Sci-Fi    0.437500
is_Drama     0.390625
dtype: float64
Sample Titles:
 The Broken Day of Justice
      The Dark City of War
The Glorious Night of Fire

=== Cluster 3 ===
Size: 57
Avg Rating: 6.44
Avg Runtime: 101.0 mins
Avg Popularity: 50.2
Top Genres:
is_Action    0.771930
is_Horror    0.456140
is_Drama

## 5. 🎞️ MP4 Animation Export (Planned)

In [ ]:
#print("A short and long MP4 animation will be generated to visualize cluster emergence over time. Coming soon!")

#conda install -c conda-forge ffmpeg
#import pandas as pd
#import numpy as np
#import matplotlib.pyplot as plt
#mport matplotlib.animation as animation
#from sklearn.preprocessing import StandardScaler
#from sklearn.decomposition import PCA
#from sklearn.manifold import TSNE
#from sklearn.cluster import KMeans

# Load your dataset (must include features and movie titles)
#df = pd.read_csv("movie_cluster_data.csv")

# Step 1: Preprocess features
#features = ["popularity", "runtime", "vote_average"] + [col for col in df.columns if col.startswith("is_")]
#X = df[features]
#X_scaled = StandardScaler().fit_transform(X)

# Step 2: Compute t-SNE embedding
#tsne = TSNE(n_components=2, perplexity=30, learning_rate="auto", init="pca", random_state=42, n_iter=500)
#tsne_result = tsne.fit_transform(X_scaled)
#df["tsne_x"] = tsne_result[:, 0]
#df["tsne_y"] = tsne_result[:, 1]

# Step 3: Run k-means clustering
#kmeans = KMeans(n_clusters=5, random_state=42)
#df["cluster"] = kmeans.fit_predict(tsne_result)

# Step 4: Set up the animation
#fig, ax = plt.subplots(figsize=(8, 6))
#scatter = ax.scatter([], [], s=60)

#def init():
#    ax.set_xlim(df["tsne_x"].min() - 5, df["tsne_x"].max() + 5)
#    ax.set_ylim(df["tsne_y"].min() - 5, df["tsne_y"].max() + 5)
#    ax.set_title("t-SNE Cluster Animation")
#    return scatter,

##def update(frame):
#    x = df["tsne_x"][:frame]
#    y = df["tsne_y"][:frame]
#    colors = df["cluster"][:frame]
#    scatter.set_offsets(np.c_[x, y])
#    scatter.set_array(colors)
#    return scatter,

# Step 5: Generate and save animations
#short_ani = animation.FuncAnimation(fig, update, frames=80, init_func=init, blit=True, interval=40)
#short_ani.save("movie_tsne_short.mp4", writer="ffmpeg", fps=24)

#long_ani = animation.FuncAnimation(fig, update, frames=len(df), init_func=init, blit=True, interval=20)
#long_ani.save("movie_tsne_full.mp4", writer="ffmpeg", fps=24)

#print("Animations saved as movie_tsne_short.mp4 and movie_tsne_full.mp4")

from IPython.display import Video

# Display short MP4 animation inline
Video("movie_tsne_short.mp4", embed=True, width=600)



A short and long MP4 animation will be generated to visualize cluster emergence over time. Coming soon!
